### build with ai training 

In [1]:
import pandas as pd
import numpy as np

# 1. Load CSV Files
symptom_df = pd.read_csv('Symptom2Disease.csv')
bed_df = pd.read_csv('Hospital_Bed_Capacity.csv')
supply_df = pd.read_csv('Suppy_Chain_Shipment_Data.csv')

# 2. Inspect Dataset Shapes & Columns
print("=== 1. SYMPTOM TO DISEASE DATASET ===")
print("Shape:", symptom_df.shape)
print("Columns:", list(symptom_df.columns))
print(symptom_df.head(2))

print("\n=== 2. HOSPITAL BED CAPACITY DATASET ===")
print("Shape:", bed_df.shape)
print("Columns:", list(bed_df.columns))
print(bed_df.head(2))

print("\n=== 3. SUPPLY CHAIN SHIPMENT DATASET ===")
print("Shape:", supply_df.shape)
print("Columns:", list(supply_df.columns))
print(supply_df.head(2))

=== 1. SYMPTOM TO DISEASE DATASET ===
Shape: (1200, 3)
Columns: ['Unnamed: 0', 'label', 'text']
   Unnamed: 0      label                                               text
0           0  Psoriasis  I have been experiencing a skin rash on my arm...
1           1  Psoriasis  My skin has been peeling, especially on my kne...

=== 2. HOSPITAL BED CAPACITY DATASET ===
Shape: (25, 7)
Columns: ['Department', 'Total_Beds', 'Free_Beds', 'Total_ICU_Beds', 'Free_ICU_Beds', 'Total_Amount_of_Beds', 'Staff_On_Duty']
                  Department  Total_Beds  Free_Beds  Total_ICU_Beds  \
0  Emergency Department (ED)         140         22              20   
1              Trauma Center         100         15              25   

   Free_ICU_Beds  Total_Amount_of_Beds  Staff_On_Duty  
0              4                   160            110  
1              5                   125             85  

=== 3. SUPPLY CHAIN SHIPMENT DATASET ===
Shape: (10324, 33)
Columns: ['id', 'project code', 'pq #', 'po / so 

In [2]:
# 1. Clean Symptom Dataset
if 'Unnamed: 0' in symptom_df.columns:
    symptom_df = symptom_df.drop(columns=['Unnamed: 0'])

symptom_df = symptom_df.dropna(subset=['text', 'label']).reset_index(drop=True)
print(f"Cleaned Symptom Dataset: {len(symptom_df)} rows")

# 2. Clean Hospital Bed Dataset
bed_df = bed_df.dropna(subset=['Department']).reset_index(drop=True)
print(f"Cleaned Bed Capacity Dataset: {len(bed_df)} rows")

# 3. Clean Supply Chain Dataset (Create delivery delay indicator)
supply_df['scheduled delivery date'] = pd.to_datetime(supply_df['scheduled delivery date'], errors='coerce')
supply_df['delivered to client date'] = pd.to_datetime(supply_df['delivered to client date'], errors='coerce')

# Calculate delay in days
supply_df['delay_days'] = (supply_df['delivered to client date'] - supply_df['scheduled delivery date']).dt.days
supply_df['is_delayed'] = (supply_df['delay_days'] > 0).astype(int)

print(f"Cleaned Supply Chain Dataset: {len(supply_df)} rows")

Cleaned Symptom Dataset: 1200 rows
Cleaned Bed Capacity Dataset: 25 rows
Cleaned Supply Chain Dataset: 10324 rows


C:\Users\Rambilas\AppData\Local\Temp\ipykernel_23848\2337432571.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  supply_df['scheduled delivery date'] = pd.to_datetime(supply_df['scheduled delivery date'], errors='coerce')
C:\Users\Rambilas\AppData\Local\Temp\ipykernel_23848\2337432571.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  supply_df['delivered to client date'] = pd.to_datetime(supply_df['delivered to client date'], errors='coerce')


In [3]:
from sklearn.model_selection import train_test_split

# Features and Labels
X = symptom_df['text']
y = symptom_df['label']

# Step 3A: First split into Train (70%) and Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Step 3B: Split Temp into Validation (15%) and Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Total dataset count: {len(X)}")
print(f"Training set count:   {len(X_train)} (70%)")
print(f"Validation set count: {len(X_val)} (15%)")
print(f"Testing set count:    {len(X_test)} (15%)")

Total dataset count: 1200
Training set count:   840 (70%)
Validation set count: 180 (15%)
Testing set count:    180 (15%)


In [4]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

print("=== CELL 4: MODEL TRAINING & EVALUATION ===")

# 1. Build NLP + Classifier Pipeline
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=5000)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# 2. Train Model on 70% Training Set
print("\nTraining model on 840 training samples...")
model_pipeline.fit(X_train, y_train)

# 3. Evaluate on 15% Validation Set
val_preds = model_pipeline.predict(X_val)
val_accuracy = accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

# 4. Final Evaluation on 15% Testing Set
test_preds = model_pipeline.predict(X_test)
test_accuracy = accuracy_score(y_test, test_preds)
print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%\n")

# Detailed Classification Report
print("Classification Report (Test Data):")
print(classification_report(y_test, test_preds))

# 5. Save Model File for Live App Deployment
joblib.dump(model_pipeline, 'disease_model.pkl')
print("Model saved successfully as 'disease_model.pkl'!")

=== CELL 4: MODEL TRAINING & EVALUATION ===

Training model on 840 training samples...
Validation Accuracy: 90.56%
Final Test Accuracy: 95.00%

Classification Report (Test Data):
                                 precision    recall  f1-score   support

                           Acne       1.00      1.00      1.00         7
                      Arthritis       1.00      1.00      1.00         7
               Bronchial Asthma       0.88      1.00      0.93         7
           Cervical spondylosis       1.00      1.00      1.00         8
                    Chicken pox       0.86      0.75      0.80         8
                    Common Cold       1.00      1.00      1.00         7
                         Dengue       0.78      0.88      0.82         8
          Dimorphic Hemorrhoids       1.00      1.00      1.00         8
               Fungal infection       0.88      1.00      0.93         7
                   Hypertension       1.00      1.00      1.00         7
                 

In [ ]:
import google.generativeai as genai

print("=== CELL 5: GOOGLE AI GEMINI INTEGRATION ===")

# 1. Set your Gemini API Key
GEMINI_API_KEY = "Your_api key"  # Replace with your actual Gemini API key
genai.configure(api_key=GEMINI_API_KEY)

# 2. End-to-End Triage Function
def run_triage_system(patient_symptoms, location="Salem, Tamil Nadu"):
    print(f"\n--- Patient Input Received ---")
    print(f"Location: {location}")
    print(f"Symptoms: {patient_symptoms}\n")
    
    # Step A: Predict Disease using Trained ML Pipeline
    predicted_disease = model_pipeline.predict([patient_symptoms])[0]
    probabilities = model_pipeline.predict_proba([patient_symptoms])[0]
    confidence = max(probabilities) * 100
    
    print(f"[ML Model Output]: Predicted Condition = {predicted_disease} ({confidence:.1f}% confidence)")
    
    # Step B: Get Hospital Bed Readiness Context
    bed_info = f"Tracked {len(bed_df)} hospital departments for bed/ICU availability."
    
    # Step C: Prompt Google Gemini API
    prompt = f"""
    You are an emergency healthcare AI assistant built for India's public health infrastructure.
    
    Patient Location: {location}
    User Reported Symptoms: {patient_symptoms}
    ML Model Diagnosis Prediction: {predicted_disease} (Confidence: {confidence:.1f}%)
    Local Hospital Status: {bed_info}
    
    Provide a structured response:
    1. Condition Explanation: Explain {predicted_disease} in very simple, easy-to-understand terms.
    2. Primary Care Advice: Give 3 clear action steps for the patient or local ASHA healthcare worker.
    3. Resource Allocation: Suggest whether emergency transport or hospital bed booking is recommended.
    """
    
    try:
        gemini_model = genai.GenerativeModel("gemini-2.5-flash")
        response = gemini_model.generate_content(prompt)
        print("\n=== GOOGLE AI GEMINI RESPONSE ===")
        print(response.text)
    except Exception as e:
        print(f"\nAPI Error: Make sure your GEMINI_API_KEY is valid. Error details: {e}")

# 3. Test Run
test_symptoms = "I have severe skin rash on my arms and legs, red itchy spots, and peeling skin."
run_triage_system(test_symptoms, location="Salem, Tamil Nadu")

=== CELL 5: GOOGLE AI GEMINI INTEGRATION ===

--- Patient Input Received ---
Location: Salem, Tamil Nadu
Symptoms: I have severe skin rash on my arms and legs, red itchy spots, and peeling skin.

[ML Model Output]: Predicted Condition = Chicken pox (25.0% confidence)

=== GOOGLE AI GEMINI RESPONSE ===
Here's a structured response based on the information provided:

1.  **Condition Explanation: Chickenpox**
    Chickenpox (called "Veesuri Ammai" or "Ammai Rogam" in Tamil) is a very common infection caused by a virus. It usually affects children but can affect adults too.
    *   **What it looks like:** You'll typically get a rash that starts as red spots, then turns into small, fluid-filled blisters that are very itchy. These blisters eventually scab over and heal. You might also have a mild fever.
    *   **How it spreads:** It's very contagious and can spread easily from person to person through coughing, sneezing, or touching the blisters.
    *   **Is it serious?** For most people, 

In [6]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
import google.generativeai as genai

# Streamlit Page Config
st.set_page_config(
    page_title="AI Health Triage & Supply Chain - India",
    page_icon="🏥",
    layout="wide"
)

st.title("🏥 Smart Health & Supply Chain Resilience System")
st.caption("Powered by Google AI Gemini & Machine Learning | Designed for India's Public Health Network")

# Load Trained Model and Datasets
@st.cache_resource
def load_resources():
    model = joblib.load('disease_model.pkl')
    symptom_df = pd.read_csv('Symptom2Disease.csv')
    bed_df = pd.read_csv('Hospital_Bed_Capacity.csv')
    supply_df = pd.read_csv('Suppy_Chain_Shipment_Data.csv')
    return model, symptom_df, bed_df, supply_df

try:
    disease_model, symptom_df, bed_df, supply_df = load_resources()
    st.sidebar.success("All ML Models & Datasets Loaded Successfully!")
except Exception as e:
    st.sidebar.error(f"Error loading resources: {e}")

# Sidebar Options
st.sidebar.header("Configuration")
api_key = st.sidebar.text_input("Enter Google Gemini API Key:", type="password")

location = st.sidebar.text_input("Patient Location / District:", value="Salem, Tamil Nadu")
language = st.sidebar.selectbox("Preferred Response Language:", ["English", "Tamil", "Hindi", "Telugu"])

# Input Section
st.subheader("1. Enter Patient Symptoms")
user_input = st.text_area(
    "Describe symptoms in plain text:",
    placeholder="Example: I have severe skin rash on my arms and legs, red itchy spots, and peeling skin."
)

if st.button("Run AI Triage & Analysis", type="primary"):
    if not api_key:
        st.warning("Please enter your Google Gemini API key in the sidebar to run the full pipeline.")
    elif not user_input.strip():
        st.warning("Please enter patient symptoms first.")
    else:
        # Step A: ML Model Prediction
        prediction = disease_model.predict([user_input])[0]
        probs = disease_model.predict_proba([user_input])[0]
        confidence = max(probs) * 100

        col1, col2, col3 = st.columns(3)
        with col1:
            st.metric("Predicted Disease", prediction)
        with col2:
            st.metric("Prediction Confidence", f"{confidence:.1f}%")
        with col3:
            st.metric("Location", location)

        # Step B: Regional Dataset Overview
        st.divider()
        st.subheader("2. Infrastructure & Supply Chain Readiness")

        bed_col, supply_col = st.columns(2)
        with bed_col:
            st.write("🏥 **Hospital Bed Capacity (Local Network)**")
            st.dataframe(bed_df[['Department', 'Total_Beds', 'Free_Beds', 'Free_ICU_Beds']].head(5))

        with supply_col:
            st.write("📦 **Supply Chain Logistics Monitoring**")
            st.dataframe(supply_df[['product group', 'sub classification', 'shipment mode', 'line item quantity']].head(5))

        # Step C: Gemini API Integration
        st.divider()
        st.subheader("3. Google AI Triage & Action Plan")

        genai.configure(api_key=api_key)

        prompt = f"""
        You are an emergency healthcare triage AI assistant supporting ASHA workers and public health officers across India.

        Patient Location: {location}
        Reported Symptoms: {user_input}
        ML Model Prediction: {prediction} (Confidence: {confidence:.1f}%)
        Response Language: {language}

        Please provide a structured report:
        1. Condition Overview: Explain {prediction} simply for local citizens. Include local terminology if applicable.
        2. Immediate Steps for ASHA Worker/Patient: 3 clear, actionable steps.
        3. Resource Allocation & Supply Logistics: Assess if emergency bed allocation or medicine shipment is required.
        """

        with st.spinner("Generating Google AI Guidance..."):
            try:
                gemini_model = genai.GenerativeModel("gemini-2.5-flash")
                response = gemini_model.generate_content(prompt)
                st.markdown(response.text)
            except Exception as e:
                st.error(f"Gemini API Call Failed: {e}")

2026-09-29 10:09:15.836 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-29 10:09:15.837 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-29 10:09:16.679 
  command:

    streamlit run C:\Users\Rambilas\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-29 10:09:16.681 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-29 10:09:16.682 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-29 10:09:16.682 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-29 10:09:16.686 Thread 'MainThread': missing ScriptRunContext! This warning ca